In [ ]:
1+1

In [3]:
from dotenv import load_dotenv
import os 
load_dotenv()

os.environ["OPENAI_API_KEY"]= os.getenv("OPENAI_API_KEY")
os.environ["groq_api_key"]= os.getenv("groq_api_key")

In [11]:

import os
import base64
import mimetypes
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()


def encode_image(image_path: str) -> tuple[str, str]:
    """Return a Base64 string and its MIME type."""
    path = Path(image_path)

    if not path.is_file():
        raise FileNotFoundError(f"Image not found: {path}")

    mime_type, _ = mimetypes.guess_type(path.name)

    supported_types = {
        "image/jpeg",
        "image/png",
        "image/webp",
        "image/gif",
    }

    if mime_type not in supported_types:
        raise ValueError(f"Unsupported image type: {mime_type}")

    image_base64 = base64.b64encode(
        path.read_bytes()
    ).decode("utf-8")

    return image_base64, mime_type


def read_image(
    image_path: str,
    
) -> str:
    """Analyze an image using OpenAI or Groq vision."""
    prompt = (
            "Read this invoice. Extract the supplier, invoice number, "
            "date, currency, subtotal, tax, and total. "
            "Mark any unreadable field as uncertain; do not guess. or extract eny other or usefull information you can "
            "find in the invoice."
        )
    try:
        model = "qwen/qwen3.8-27b"
        api_key = os.getenv("GROQ_API_KEY")
        base_url = "https://api.groq.com/openai/v1"
        if not model:
            raise ValueError(
                "Pass a currently supported Groq vision model."
            )

    except ValueError as e:
        raise ValueError("Provider must be 'openai' or 'groq'.")


    image_base64, mime_type = encode_image(image_path)

    client = OpenAI(
        api_key=api_key,
        **({"base_url": base_url} if base_url else {}),
    )

    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": (
                                f"data:{mime_type};base64,"
                                f"{image_base64}"
                            )
                        },
                    },
                ],
            }
        ],
    )

    return response.choices[0].message.content or ""


In [12]:
result = read_image(
        image_path=r"C:\dev\accountant_agent_v3\laboratoire\xx.jpg",
    )

print(result)

Based on the documents provided (which appear to be multiple views of the same receipt), here is the extracted information:

*   **Supplier:** MACONFIG (Address: Quartier Industriel Massar Immeuble 836, MARRAKECH 40000, Maroc)
*   **Invoice Number:** Commande # S45620
*   **Date:** 06/01/2026
*   **Currency:** DH (Moroccan Dirham)
*   **Subtotal:** 549,00 DH
*   **Tax:** 35,00 DH
*   **Total:** 584,00 DH

**Additional Useful Information:**

*   **Customer Name:** Baher Illas
*   **Customer Address:** Settati hay al kamal, Maroc
*   **Salesperson (Vendeur):** Kaoutar IMARGHNE
*   **Product:** Lexar NM620 M.2 PCIe NVMMe 570 1TB (1,00 Units @ 549,00 DH)
*   **Internal Reference:** #141498 ORD-26-01-5-213615
*   **Note:** [Livr] Livraison (Free shipping/Delivery)


In [ ]:
import os
from supabase import create_client
from dotenv import load_dotenv

load_dotenv()

url = os.getenv("SUPABASE_URL")
key = os.getenv("SUPABASE_KEY")
supabase = create_client(url, key)


#insert_1 = supabase.table("User").insert({'email': 'markterstegebn@gmail.com', 'password': 'hcbudhcidjdsuh'}).execute()
#update_1 = supabase.table("User").update({"password": "updated pass"}).eq("uuid", 2).execute()
#delete_1 = supabase.table("User").delete().eq("uuid",3).execute()
#data = supabase.table("User").select("*").execute()
data = supabase.table("Company").select("id").execute()
print(data.data)